<a href="https://colab.research.google.com/github/jagan631/AI-Powered-Document-Chatbot/blob/main/RAG_Document_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-chroma \
langchain-huggingface \
chromadb \
sentence-transformers \
transformers \
accelerate \
pymupdf \
langchain-text-splitters


In [8]:
from google.colab import files

from langchain_community.document_loaders import PyMuPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import pipeline

from langchain_huggingface import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [10]:
uploaded = files.upload()

Saving Part2.pdf to Part2.pdf


In [11]:
loader = PyMuPDFLoader("Part2.pdf")

documents = loader.load()

print(f"Total Pages: {len(documents)}")

Total Pages: 6


In [12]:
print(documents[0].page_content)

Sample-1: Easy 
You have an array A of integers with n elements. There 
are q queries to process and each query consists of four 
integers: l, r, x, and y. 
 For the subarray of A ranging from index l to r, you need to 
assign a sequence of integers for each subsequent element. 
The sequence should start from x and increase by y. This means: 
•
A[l] will be assigned the value of x. 
•
A[l+1] will be assigned the value of x + y. 
•
A[l+2] will be assigned the value of x + 2*y. 
•
Continuing this pattern, A[l+i] will be assigned the 
value of x + i*y, where i ranges from 0 to (r - l). 
Find the sum of all integers in A after processing all queries. 
Since answer can be large, return it modulo 109+7. 
Input Format 
1.
The first line contains an integer, n, denoting the 
number of elements in A. 
2.
Each line i of the n subsequent lines (where 0 ≤ i < n) 
contains an integer describing A[i]. 
3.
The next line contains an integer, q, denoting the 
number of rows in queries. 
4.
Each line i 

In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 31


In [14]:
print(chunks[0].page_content)

Sample-1: Easy 
You have an array A of integers with n elements. There 
are q queries to process and each query consists of four 
integers: l, r, x, and y. 
 For the subarray of A ranging from index l to r, you need to 
assign a sequence of integers for each subsequent element. 
The sequence should start from x and increase by y. This means: 
•
A[l] will be assigned the value of x. 
•
A[l+1] will be assigned the value of x + y. 
•
A[l+2] will be assigned the value of x + 2*y. 
•


In [15]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [17]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

In [18]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [19]:
docs = retriever.invoke(
    "what is the easy question"
)

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}")
    print("=" * 50)
    print(doc.page_content)


Chunk 1
Sample-1: Easy 
You have an array A of integers with n elements. There 
are q queries to process and each query consists of four 
integers: l, r, x, and y. 
 For the subarray of A ranging from index l to r, you need to 
assign a sequence of integers for each subsequent element. 
The sequence should start from x and increase by y. This means: 
•
A[l] will be assigned the value of x. 
•
A[l+1] will be assigned the value of x + y. 
•
A[l+2] will be assigned the value of x + 2*y. 
•

Chunk 2
l = 0, r = 2, x = 1, y = 2
A[0] = 1 
A[1] = 3 
A[2] = 5 
So, A = [1, 3, 5, 3, 0] 
for query 2: 
l = 0, r = 1, x = 6, y = 5
A[0] = 6 
A[1] = 11 
So, A = [6, 11, 5, 3, 0] 
for query 3: 
l = 2, r = 3, x = 8, y = 0
A[2] = 8 
A[3] = 8 
So, A = [6, 11, 8, 8, 0] 
for query 4: 
l = 2, r = 4, x = 9, y = 6
A = [6, 11, 9, 15, 21] 
for query 5: 
l = 3, r = 4, x = 8, y = 9
A = [6, 11, 9, 8, 17] 
Hence, answer is 6+11+9+8+17 = 51 
Sample Input 2 
5 
3 
9 
2 
5 
4 
5 
1 2 6 3 
1 2 2 8 
1 2 5 5 
1 3 1 8

Chun

In [20]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [21]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [22]:
llm = HuggingFacePipeline(
    pipeline=pipe
)

In [23]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context, reply with:

"I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [24]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
question = "what is hard question"

answer = rag_chain.invoke(question)

print(answer)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context, reply with:

"I don't know."

Context:
Sample 3 : Hard 
You are given a tree with n nodes rooted at node 1. You are also 
given an array color representing the colour of each node in the 
tree. 
A set of nodes is beautiful if it satisfies the following conditions: 
• 
All nodes in the set have different colors. 
• 
For any pair of nodes (u, v), either u is the ancestor of v or v is 
the ancestor of u within the tree. 
You're given q queries where each query provides an 
integer s representing a node in the tree.

for query 3: 
    s = 3, means we need to select beautiful set of maximum size in 
the subtree of 3. 
    we can select nodes {3, 5} to form beautiful set of maximum 
size. 
    so, answer for this query is 2. 
 
Hence, answer is 1 + 2 + 2 = 5. 
 
Sample input-2: 
5 
0 
1 
1 
2 
2 
1 
5 
4 
5 
2 
3 
5 
4 
3 
 
Sample output-2: 
3

for query 4: 
    s = 1, me

In [27]:
while True:
    question = input("Ask a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nAnswer:\n")
    print(answer)
    print("-" * 80)

Ask a question (type 'exit' to quit): quit


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:


You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context, reply with:

"I don't know."

Context:
prior permission of Infosys Limited and/ or any named intellectual property rights holders under this document. 
Infosys.com | NYSE: INFY 
Stay Connected

3 4 9 1 
2 3 1 0 
 
Sample output 3 
16 
 
Explanation 3 
Here, n = 5 
A = [0, 1, 0, 0, 1] 
q = 5 
queries = [[1, 2, 7, 7], [0, 1, 3, 6], [1, 
1, 1, 1], [3, 4, 9, 1], [2, 3, 1, 0]] 
 
for query 1: 
l = 1, r = 2, x = 7, y = 7 
A = [0, 7, 14, 0, 1] 
 
 
 
for query 2: 
l = 0, r = 1, x = 3, y = 6 
A = [3, 9, 14, 0, 1] 
 
for query 3: 
l = 1, r = 1, x = 1, y = 1 
A = [3, 1, 14, 0, 1] 
 
for query 4: 
l = 3, r = 4, x = 9, y = 1 
A = [3, 1, 14, 9, 10] 
 
for query 5: 
l = 2, r = 3, x = 1, y = 0

x and y for the i-th query. 
Constraints 
•
1 <= 𝒏 <= 10!
•
0 <= 𝑨[𝒊] <= 10"
•
1 ≤ 𝒒 ≤ 10!
•
0 <= 𝒒𝒖𝒆𝒓𝒊𝒆𝒔[𝒊][𝒋] <= 10!
          Sample Input 1 
5 
5 
5 
0 
3 
0 
5 
0 2 1 2 
0 1 6